## installing dependencies 
Rag pipeline:
document loading 
embedding model 
vector database 
llm model



In [36]:

import langchain


print("Installing dependencies")


%pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface langchain-groq langchain-google-genai langchain-openai langchain-core faiss-cpu pypdf sentence-transformers transformers torch huggingface_hub groq langsmith python-dotenv tiktoken 

print("Installation completed")



Installing dependencies
Note: you may need to restart the kernel to use updated packages.
Installation completed


In [26]:
LLM_Provider='Gnani AI' 
LLM_Model='Evonv3.3' 
LLM_API_KEY='GNANI_API_KEY'


CORPUS_PATH="./zyro-dynamics-hr-corpus"

In [9]:
import os 
from langchain_community.document_loaders import PyPDFDirectoryLoader  
from langchain_text_splitters import RecursiveCharacterTextSplitter   




In [10]:
loader=PyPDFDirectoryLoader(CORPUS_PATH) 
documents=loader.load() 
print(f"Loaded {len(documents)} documents")

Loaded 39 documents


In [11]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=800, 
    chunk_overlap=100
)  
chunks=splitter.split_documents(documents) 
print(f"created {len(chunks)} chunks")

created 107 chunks


HuggingFaceEmbeddings

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings 
model=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
) 
print("Embeddings done!")

c:\Users\markala ugala\OneDrive\Desktop\intelligent rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1653.77it/s]


Embeddings done!


VectorDB+Retrival

FAISS vector DB

In [13]:
from langchain_community.vectorstores import FAISS  
vectorstore=FAISS.from_documents(chunks, model)
retriver=vectorstore.as_retriever(search_kwargs={"k":3})
print("done!")

done!


In [28]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GNANI_API_KEY")

print("API key loaded:", bool(api_key))

API key loaded: True


In [30]:
## LLM initialization 

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

GNANI_API_KEY = os.getenv("GNANI_API_KEY")

if not GNANI_API_KEY:
    raise ValueError("Gnani API key not found")

llm = ChatOpenAI(
    model="Ebon v3.3",
    api_key=GNANI_API_KEY,
    temperature=0.7,
    max_tokens=500
)

print("LLM initialized")

LLM initialized


RAG CHAIN

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser 
from langsmith import traceable
RAG_PROMPT=ChatPromptTemplate.from_template(
    """ you are HR assistant .Answer the question using ONLY the context below.If 
    the answer isn't in the context,say you don't have that information. 
    
    context: {context},
    question: {question}"""
)
def format_docs(docs): 
    return "\n".join(d.page_content for d in docs)

@traceable(name="rag_chain")
def rag_chain(question:str):
    docs=retriver.invoke(question) 
    context=format_docs(docs)
    chain=RAG_PROMPT | llm_model | StrOutputParser()  
    answer=chain.invoke({"context":context,"question":question})   
    return {"answer":answer}


Guardians 
(To ensure no context is used beyond the corpus)

In [34]:
GUARDIAL_PROMPT=ChatPromptTemplate.from_template(""" 
you are a scope classifier for an HR assistant.Decide whether the question
below is something an HR assistant should answer (company leave policy,reimbrusement,code of conduct,
or other internal HR topics) or other something out of scope (general knowledge,coding help,unrelated small talk,etc).
respond with exactly one word : IN_SCOPE or OUT_OF_SCOPE.
Question:{question}
""")
REFUSAL_MESSAGE=(
    "I'm an HR assistant and can only help with questions about company HR" 
    "policies (leave,reimbursement,code of conduct,etc).I don't have" 
    "Information to answer that question."
)
def ask_bot(question: str):

    if llm is None:
        return {
            "answer": "Please initialize the Gnani LLM first.",
            "sources": []
        }

    try:
        guardial_chain = GUARDIAL_PROMPT | llm | StrOutputParser()

        verdict = guardial_chain.invoke({
            "context": "",
            "question": question
        })

    except Exception as exc:
        return {
            "answer": f"Model error: {exc}",
            "sources": []
        }

    if "OUT_OF_SCOPE" in verdict:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": []
        }

    return rag_chain.invoke(question)

In [35]:
test_question=[
    "How many casual leaves do I get per year?",
    "What is the internet reimbursement limit?",
    "what's the capital of France?",
]
for i,q in enumerate(test_question):

    result=ask_bot(q) 
    print(f"Q{i}:{q}")
    print(f"A{i}:{result['answer']}")
    if result['sources']:
        print(f"sources:{[s.metadata['source'] for s in result['sources']]}") 
        print("-"*60)

Q0:How many casual leaves do I get per year?
A0:Model error: Connection error.
Q1:What is the internet reimbursement limit?
A1:Model error: Connection error.
Q2:what's the capital of France?
A2:Model error: Connection error.
